In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
import opendatasets as od

url = 'https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017'
od.download(url)

Skipping, found downloaded files in "./international-football-results-from-1872-to-2017" (use force=True to force download)


In [ ]:
!ls

!mv international-football-results-from-1872-to-2017 intl_results

international-football-results-from-1872-to-2017
intl_results
World Cup.ipynb


In [ ]:
df = pd.read_csv('intl_results/results.csv')
df.shape

(49477, 9)

In [9]:
df.columns.tolist()

['date',
 'home_team',
 'away_team',
 'home_score',
 'away_score',
 'tournament',
 'city',
 'country',
 'neutral']

In [10]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [11]:
df.dtypes

date           object
home_team      object
away_team      object
home_score    float64
away_score    float64
tournament     object
city           object
country        object
neutral          bool
dtype: object

In [12]:
# Converting the 'date' column from text to actual datetime objects
df['date'] = pd.to_datetime(df['date'])


# Checking the earliest and latest match dates in the dataset
print(df['date'].min(), df['date'].max())

1872-11-30 00:00:00 2026-06-27 00:00:00


In [13]:
# Count how many distinct team names appear as home teams vs away teams
print(f'Unique Home teams: {df["home_team"].nunique()}')
print(f'Unique Away teams: {df["away_team"].nunique()}')


Unique Home teams: 327
Unique Away teams: 321


In [14]:
# Counting how many different tournament types exist (World Cup, Friendly,
# qualifiers, continental championships, etc.)
print("Unique tournament types:", df['tournament'].nunique())

Unique tournament types: 200


In [15]:
# Checking for missing values in every column
print(df.isna().sum())

date           0
home_team      0
away_team      0
home_score    70
away_score    70
tournament     0
city           0
country        0
neutral        0
dtype: int64


The dataset contains 49,477 international matches with no missing values
in team names, dates, or tournament info. Only 70 matches are missing
`home_score`/`away_score` (likely abandoned or unrecorded matches)
these will be dropped since they can't be used for training

In [16]:
# Drop the unscored matches
df = df.dropna(subset=['home_score', 'away_score'])

df['home_score'] = df['home_score'].astype(int)
df['away_score'] = df['away_score'].astype(int)

print(df.shape)

(49407, 9)


In [17]:
# Checking the amount of matches per de
df['year'] = df['date'].dt.year
print(df['year'].value_counts().sort_index().tail(30))


year
1997     907
1998     761
1999     776
2000    1040
2001    1032
2002     768
2003     947
2004    1079
2005     804
2006     842
2007     988
2008    1101
2009     925
2010     863
2011    1119
2012    1025
2013     963
2014     856
2015    1039
2016     920
2017     924
2018     929
2019    1149
2020     347
2021    1115
2022     970
2023    1054
2024    1231
2025    1002
2026     313
Name: count, dtype: int64


In [51]:
df = df[(df['year'] >= 2002) & (df['date'] < '2026-06-11')].copy()
print(df.shape)
print(f"{df['home_team'].nunique()} unique home teams")

(23271, 19)
314 unique home teams


In [19]:
# Loading previous names and mapping with current
former = pd.read_csv('intl_results/former_names.csv')
print(former.shape)
print(former.head(10))
print(former.columns.tolist())

(36, 4)
          current                                former  start_date  \
0           Benin                               Dahomey  1959-11-08   
1    Burkina Faso                           Upper Volta  1960-04-14   
2         Curaçao                  Netherlands Antilles  1957-03-03   
3  Czechoslovakia                               Bohemia  1903-04-05   
4  Czechoslovakia                   Bohemia and Moravia  1939-01-01   
5  Czechoslovakia  Representation of Czechs and Slovaks  1993-03-24   
6        DR Congo                         Belgian Congo  1948-05-25   
7        DR Congo                    Congo-Léopoldville  1963-04-12   
8        DR Congo                        Congo-Kinshasa  1965-01-09   
9        DR Congo                                 Zaïre  1971-01-10   

     end_date  
0  1975-11-30  
1  1984-08-04  
2  2010-10-10  
3  1919-01-01  
4  1945-05-01  
5  1993-11-17  
6  1956-01-02  
7  1964-07-19  
8  1970-11-24  
9  1997-04-27  
['current', 'former', 'start_date'

In [20]:
# Mapping old names into new current names
name_map = dict(zip(former['former'], former['current']))

# Renaming home and away teams
df['home_team'] = df['home_team'].replace(name_map)
df['away_team'] = df['away_team'].replace(name_map)

# Checking reduction in unique names
print(df['home_team'].nunique())

314


In [21]:
wc_teams = [
    "Mexico", "South Africa", "South Korea", "Czechia",
    "Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland",
    "Brazil", "Morocco", "Haiti", "Scotland",
    "United States", "Paraguay", "Australia", "Türkiye",
    "Germany", "Curaçao", "Côte d'Ivoire", "Ecuador",
    "Netherlands", "Japan", "Sweden", "Tunisia",
    "Belgium", "Egypt", "IR Iran", "New Zealand",
    "Spain", "Cabo Verde", "Saudi Arabia", "Uruguay",
    "France", "Senegal", "Iraq", "Norway",
    "Argentina", "Algeria", "Austria", "Jordan",
    "Portugal", "Congo DR", "Uzbekistan", "Colombia",
    "England", "Croatia", "Ghana", "Panama",
]

# All team names that are present
all_teams_in_df = set(df['home_team']).union(set(df['away_team']))
missing = [t for t in wc_teams if t not in all_teams_in_df]
print(missing)

['Czechia', 'Türkiye', "Côte d'Ivoire", 'IR Iran', 'Cabo Verde', 'Congo DR']


In [22]:
print([t for t in all_teams_in_df if 'Czech' in t])
print([t for t in all_teams_in_df if 'Turk' in t or 'Türk' in t])
print([t for t in all_teams_in_df if 'Ivoire' in t or 'Ivory' in t])
print([t for t in all_teams_in_df if 'Iran' in t])
print([t for t in all_teams_in_df if 'Verde' in t])
print([t for t in all_teams_in_df if 'Congo' in t])

['Czech Republic']
['Turks and Caicos Islands', 'Turkmenistan', 'Turkey', 'East Turkestan']
['Ivory Coast']
['Iran']
['Cape Verde']
['Congo', 'DR Congo']


In [23]:
manual_map = {
    "Czech Republic": "Czechia",
    "Turkey": "Türkiye",
    "Ivory Coast": "Côte d'Ivoire",
    "Iran": "IR Iran",
    "Cape Verde": "Cabo Verde",
    "DR Congo": "Congo DR",
}

# Apply renames for missing countries
df['home_team'] = df['home_team'].replace(manual_map)  
df['away_team'] = df['away_team'].replace(manual_map)

In [24]:
# Updating the set and checking for missing countries again
all_teams_in_df = set(df['home_team']).union(set(df['away_team']))  
missing = [t for t in wc_teams if t not in all_teams_in_df] 
print("Still missing:", missing)

Still missing: []


Applied `former_names.csv` mappings plus a manual mapping to fix
remaining naming convention differences (e.g. "Turkey" -> "Türkiye",
"Iran" -> "IR Iran", "DR Congo" -> "Congo DR"). All 48 World Cup 2026
teams now have consistent names matching the dataset.

## Next step: Team Strength via Elo Ratings

Before bringing in external FIFA ranking data, I'll compute a rolling
Elo rating for every team directly from match results. This gives a
continuously updating strength score and captures "form" naturally.

In [25]:
df = df.sort_values('date').reset_index(drop=True)

In [26]:
elo = {} # Current ratings per team
K = 20 # How much rating move per match
HOME_ADV = 60 # Home advantage in Elo points

home_elo_before = []
away_elo_before = []

for idx, row in df.iterrows():
    home, away = row['home_team'], row['away_team']

    r_home = elo.get(home, 1500)
    r_away = elo.get(away, 1500)

    home_elo_before.append(r_home)
    away_elo_before.append(r_away)

    # Adjusting home advantage
    adj_home = r_home + (0 if row['neutral'] else HOME_ADV)

    # Expected score
    exp_home = 1 / (1 + 10 ** ((r_away - adj_home) / 400))


    if row['home_score'] > row['away_score']:
        actual_home = 1
    elif row['home_score'] == row['away_score']:
        actual_home = 0.5
    else:
        actual_home = 0

    # Updating the ratings
    elo[home] = r_home + K * (actual_home - exp_home)
    elo[away] = r_away + K * ((1 - actual_home) - (1 - exp_home))

df['home_elo'] = home_elo_before
df['away_elo'] = away_elo_before

In [27]:
print(df[['date','home_team','away_team','home_elo','away_elo']].tail(10))


            date    home_team                 away_team     home_elo  \
23263 2026-06-09      Armenia                   Moldova  1384.917277   
23264 2026-06-09    Argentina                   Iceland  1943.142780   
23265 2026-06-09       Angola  Central African Republic  1541.040643   
23266 2026-06-09      Hungary                Kazakhstan  1636.409656   
23267 2026-06-10     Portugal                   Nigeria  1835.832550   
23268 2026-06-10      Bolivia                   Algeria  1543.038450   
23269 2026-06-10      England                Costa Rica  1823.008002   
23270 2026-06-10  Afghanistan                  Pakistan  1367.614392   
23271 2026-06-11       Mexico              South Africa  1787.725836   
23272 2026-06-11  South Korea                   Czechia  1761.986357   

          away_elo  
23263  1295.781683  
23264  1524.174236  
23265  1417.561598  
23266  1416.100611  
23267  1716.705247  
23268  1776.886661  
23269  1638.484204  
23270  1277.141310  
23271  1612.023641

A rolling Elo rating was computed for every team based on chronological
match results (2002-2026), starting from a baseline of 1500, with a
home advantage adjustment (+60 for non-neutral venues) and K=20.
Pre-match ratings were stored as `home_elo` / `away_elo`.

In [28]:
# Calculating the elo difference
df['elo_diff'] = df['home_elo'] - df['away_elo']

In [29]:
from collections import defaultdict, deque

# Rolling window size
N = 5

history = defaultdict(lambda: deque(maxlen=N))
home_form_gf, home_form_ga = [], []
away_form_gf, away_form_ga = [], []

for idx, row in df.iterrows():
    home, away = row['home_team'], row['away_team']

    # Average goals for/against over last N matches
    h_hist = history[home]
    a_hist = history[away]

    home_form_gf.append(sum(g[0] for g in h_hist) / len(h_hist) if h_hist else 0)
    home_form_ga.append(sum(g[1] for g in h_hist) / len(h_hist) if h_hist else 0)
    away_form_gf.append(sum(g[0] for g in a_hist) / len(a_hist) if a_hist else 0)
    away_form_ga.append(sum(g[1] for g in a_hist) / len(a_hist) if a_hist else 0)

    history[home].append((row['home_score'], row['away_score']))
    history[away].append((row['away_score'], row['home_score']))

df['home_form_gf'] = home_form_gf
df['home_form_ga'] = home_form_ga
df['away_form_gf'] = away_form_gf
df['away_form_ga'] = away_form_ga

df[['date','home_team','away_team','home_form_gf','home_form_ga','away_form_gf','away_form_ga']].tail(5)


,date,home_team,away_team,home_form_gf,home_form_ga,away_form_gf,away_form_ga
23268,2026-06-10,Bolivia,Algeria,1.2,1.6,1.8,0.4
23269,2026-06-10,England,Costa Rica,1.2,0.4,0.6,2.2
23270,2026-06-10,Afghanistan,Pakistan,0.4,0.8,1.2,1.2
23271,2026-06-11,Mexico,South Africa,1.8,0.4,0.8,1.0
23272,2026-06-11,South Korea,Czechia,1.4,1.0,3.4,1.2


## Next step: Match Importance Feature

The `tournament` column has many specific categories (World Cup,
qualifiers, continental championships, friendlies, etc.). I'll group
these into a small set of importance levels, since competitive matches
tend to be played more intensely than friendlies.

In [30]:
# Most common tournament types
df['tournament'].value_counts().head(30)

tournament
Friendly                                7811
FIFA World Cup qualification            5066
UEFA Euro qualification                 1532
African Cup of Nations qualification    1302
UEFA Nations League                      658
African Cup of Nations                   493
AFC Asian Cup qualification              470
CONCACAF Nations League                  422
FIFA World Cup                           386
Gold Cup                                 340
CECAFA Cup                               312
COSAFA Cup                               302
CFU Caribbean Cup qualification          264
Island Games                             259
UEFA Euro                                246
AFF Championship                         231
AFC Asian Cup                            230
Copa América                             222
Gulf Cup                                 189
SAFF Cup                                 135
EAFF Championship                        130
CONIFA World Football Cup                101

In [31]:
importance_map = {
    'FIFA World Cup': 3,
    'FIFA World Cup qualification': 2,
    'UEFA Euro': 3,
    'UEFA Euro qualification': 2,
    'Copa América': 3,
    'African Cup of Nations': 3,
    'African Cup of Nations qualification': 2,
    'AFC Asian Cup': 3,
    'AFC Asian Cup qualification': 2,
    'CONCACAF Nations League': 2,
    'UEFA Nations League': 2,
    'Gold Cup': 2,
    'Confederations Cup': 3,
    'Friendly': 0,
}

df['match_importance'] = df['tournament'].map(importance_map).fillna(1)
print(df['match_importance'].value_counts())

match_importance
2.0    9790
0.0    7811
1.0    4015
3.0    1657
Name: count, dtype: int64


Core features for each match
- `home_elo`, `away_elo` (and `elo_diff`)
- `home_form_gf`, `home_form_ga`, `away_form_gf`, `away_form_ga`
- `neutral` (boolean)
- `match_importance` (0-3)

In [32]:
# 3-class outcome label for baseline classifier 
# 0 = away win, 1 = draw, 2 = home win

df['result'] = (df['home_score'] > df['away_score']).astype(int) * 2 + \
                (df['home_score'] == df['away_score']).astype(int)

df['result'].value_counts()

result
2    11163
0     6687
1     5423
Name: count, dtype: int64

In [33]:
# Building the final dataframe

feature_cols = [
    'elo_diff', 'home_elo', 'away_elo',
    'home_form_gf', 'home_form_ga',
    'away_form_gf', 'away_form_ga',
    'neutral', 'match_importance'
]

target_cols = ['home_score', 'away_score', 'result']

model_df = df[feature_cols + target_cols + ['date', 'home_team', 'away_team']].copy()

model_df.shape

(23273, 15)

In [34]:
model_df.isna().sum()

elo_diff            0
home_elo            0
away_elo            0
home_form_gf        0
home_form_ga        0
away_form_gf        0
away_form_ga        0
neutral             0
match_importance    0
home_score          0
away_score          0
result              0
date                0
home_team           0
away_team           0
dtype: int64

### Train Test Split

Since this is time-series data, I'll split the data split chronologically rather than randomly - training on older matches and testing on more recent ones

In [35]:
split_date = model_df['date'].quantile(0.85)
split_date

Timestamp('2023-03-24 00:00:00')

In [36]:
train = model_df[model_df['date'] < split_date]
test = model_df[model_df['date'] >= split_date]

In [37]:
X_train = train[feature_cols]
X_test = test[feature_cols]

y_train_result = train['result']
y_test_result = test['result']

y_train_home_score = train['home_score']
y_test_home_score = test['home_score']

y_train_away_score = train['away_score']
y_test_away_score = test['away_score']

In [38]:
train.shape, test.shape

((19751, 15), (3522, 15))

In [54]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss, classification_report, f1_score
from xgboost import XGBClassifier

def compare_models(X_train, X_test, y_train, y_test):

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    models = {
        'Logistic Regression': (LogisticRegression(max_iter=1000, multi_class='multinomial'), True),
        'Random Forest': (RandomForestClassifier(n_estimators=200, random_state=42), False),
        'Gradient Boosting': (GradientBoostingClassifier(random_state=42), False),
        'XGBoost': (XGBClassifier(eval_metric='mlogloss', random_state=42), False)
    }

    results = []

    for name, (model, needs_scaling) in models.items():
        Xtr = X_train_scaled if needs_scaling else X_train
        Xte = X_test_scaled if needs_scaling else X_test

        model.fit(Xtr, y_train)
        y_pred = model.predict(Xte)
        probs = model.predict_proba(Xte)

        acc = accuracy_score(y_test, y_pred)
        ll = log_loss(y_test, probs)
        macro_f1 = f1_score(y_test, y_pred, average='macro')

        results.append({
            'Model': name,
            'Accuracy': round(acc, 4),
            'Log Loss': round(ll, 4),
            'Macro F1': round(macro_f1, 4)
        })

        print(f"\n=== {name} ===")
        print(classification_report(y_test, y_pred, target_names=['Away Win','Draw','Home Win']))

    return pd.DataFrame(results).sort_values('Accuracy', ascending=False)

In [55]:
results_df = compare_models(X_train, X_test, y_train_result, y_test_result)
results_df


=== Logistic Regression ===
              precision    recall  f1-score   support

    Away Win       0.57      0.64      0.60      1048
        Draw       0.55      0.01      0.01       808
    Home Win       0.62      0.87      0.73      1666

    accuracy                           0.60      3522
   macro avg       0.58      0.51      0.45      3522
weighted avg       0.59      0.60      0.53      3522


=== Random Forest ===
              precision    recall  f1-score   support

    Away Win       0.57      0.61      0.59      1048
        Draw       0.29      0.09      0.14       808
    Home Win       0.63      0.81      0.71      1666

    accuracy                           0.59      3522
   macro avg       0.50      0.50      0.48      3522
weighted avg       0.53      0.59      0.54      3522


=== Gradient Boosting ===
              precision    recall  f1-score   support

    Away Win       0.57      0.65      0.60      1048
        Draw       0.27      0.02      0.03       

,Model,Accuracy,Log Loss,Macro F1
0,Logistic Regression,0.6042,0.8751,0.4477
2,Gradient Boosting,0.6011,0.8786,0.4532
3,XGBoost,0.5906,0.9279,0.4827
1,Random Forest,0.5855,0.9155,0.4777


All models score ~57-60% accuracy, but **Draw recall is only 2-12%**
across all four which means they almost never correctly predict a draw. The gap
between macro F1 (0.45-0.48) and weighted F1 (0.53-0.54) shows
performance is propped up by the majority class (Home Win), while
Draw is essentially ignored.

This happens because draws don't have a distinct region in feature
space and they can occur between two strong teams or two weak teams alike

### Poisson Goal Regression Approach

In [56]:
from xgboost import XGBRegressor
home_goal_model = XGBRegressor(objective='count:poisson', 
                                n_estimators=500, max_depth=3,
                                learning_rate=0.03, subsample=0.8,
                                colsample_bytree=0.8, reg_lambda=2,
                                random_state=42)

away_goal_model = XGBRegressor(objective='count:poisson', 
                                n_estimators=500, max_depth=3,
                                learning_rate=0.03, subsample=0.8,
                                colsample_bytree=0.8, reg_lambda=2,
                                random_state=42)

In [58]:
home_goal_model.fit(X_train, y_train_home_score)
away_goal_model.fit(X_train, y_train_away_score)

home_ypred = home_goal_model.predict(X_test)
away_ypred = away_goal_model.predict(X_test)

home_ypred = np.clip(home_ypred, 0.01, None)
away_ypred = np.clip(away_ypred, 0.01, None)

In [59]:
from sklearn.metrics import mean_absolute_error, mean_poisson_deviance

print(f'Home goals - MAE: {mean_absolute_error(y_test_home_score, home_ypred)}')
print(f'Home goals - Mean Poisson Deviance: {mean_poisson_deviance(y_test_home_score, home_ypred)}')
print(f'Away goals - MAE: {mean_absolute_error(y_test_away_score, away_ypred)}')
print(f'Away goals - Mean Poisson Deviance: {mean_poisson_deviance(y_test_away_score, away_ypred)}')


Home goals - MAE: 1.0569228436681508
Home goals - Mean Poisson Deviance: 1.2465213573218072
Away goals - MAE: 0.8619482438230738
Away goals - Mean Poisson Deviance: 1.1777722034291302


In [60]:
print(f'Avg actual home goals: {y_test_home_score.mean()} | Avg predicted: {home_ypred.mean()}')
print(f'Avg actual away goals: {y_test_away_score.mean()} | Avg predicted: {away_ypred.mean()}')

Avg actual home goals: 1.6345826235093697 | Avg predicted: 1.6693938970565796
Avg actual away goals: 1.147075525269733 | Avg predicted: 1.1204811334609985


In [64]:
current_state = {}

for team in wc_teams:
    h_hist = history[team]

    if h_hist:
        form_gf = sum(g[0] for g in h_hist) / len(h_hist)
        form_ga = sum(g[1] for g in h_hist) / len(h_hist)
    else:
        form_gf, form_ga = 0

    current_state[team] = {
        'elo': elo.get(team, 1500),
        'form_gf': form_gf,
        'form_ga': form_ga
    }

for t in ['Argentina', 'France', 'Brazil', 'Haiti']:
    print(t, current_state[t])

Argentina {'elo': 1944.7883608429909, 'form_gf': 2.8, 'form_ga': 0.2}
France {'elo': 1873.2754309970658, 'form_gf': 2.4, 'form_ga': 1.2}
Brazil {'elo': 1876.949448482813, 'form_gf': 2.6, 'form_ga': 1.4}
Haiti {'elo': 1603.4416011594865, 'form_gf': 1.6, 'form_ga': 0.8}


In [62]:
from scipy.stats import poisson

def predict_score_probabilities(home_xg, away_xg, max_goals=8):
    probs = np.zeros((max_goals + 1, max_goals + 1))
    for i in range(max_goals + 1):
        for j in range(max_goals + 1):
            probs[i, j] = poisson.pmf(i, home_xg) * poisson.pmf(j, away_xg)

    # normalizing the sum
    probs /= probs.sum()

    p_home_win = np.tril(probs, -1).sum() # home > away
    p_draw = np.trace(probs) # home == away
    p_away_win = np.triu(probs, 1).sum() # away > home

    most_likely_score = np.unravel_index(np.argmax(probs), probs.shape)
    return p_home_win, p_draw, p_away_win, most_likely_score, probs



In [65]:
def build_feature_row(team_a, team_b, neutral=True, match_importance=3):
    a = current_state[team_a]
    b = current_state[team_b]

    row = {
        'elo_diff': a['elo'] - b['elo'],
        'home_elo': a['elo'],
        'away_elo': b['elo'],
        'home_form_gf': a['form_gf'],
        'home_form_ga': a['form_ga'],
        'away_form_gf': b['form_gf'],
        'away_form_ga': b['form_ga'],
        'neutral': neutral,
        'match_importance': match_importance
    }

    return pd.DataFrame([row])[feature_cols]

In [66]:
def predict_match(team_a, team_b, neutral=True, match_importance=3):
    X_row = build_feature_row(team_a, team_b, neutral, match_importance)

    lam_a = home_goal_model.predict(X_row)[0]
    lam_b = away_goal_model.predict(X_row)[0]

    p_a, p_draw, p_b, best_score, prob_matrix = predict_score_probabilities(lam_a, lam_b)

    print(f"\n{'='*45}")
    print(f"  {team_a} vs {team_b}")
    print(f"{'='*45}")
    print(f"  Expected goals     : {team_a} {lam_a:.2f} | {team_b} {lam_b:.2f}")
    print(f"  Predicted total    : {lam_a + lam_b:.2f} goals")
    print(f"  Most likely score  : {best_score[0]}-{best_score[1]}")
    print(f"  {team_a} win      : {p_a*100:.1f}%")
    print(f"  Draw               : {p_draw*100:.1f}%")
    print(f"  {team_b} win      : {p_b*100:.1f}%")

    return {
        "team_a": team_a, "team_b": team_b,
        "lambda_a": round(lam_a, 3),
        "lambda_b": round(lam_b, 3),
        "total_goals": round(lam_a + lam_b, 3),
        "most_likely_score": f"{best_score[0]}-{best_score[1]}",
        "p_team_a_win": round(p_a * 100, 1),
        "p_draw": round(p_draw * 100, 1),
        "p_team_b_win": round(p_b * 100, 1),
        "prob_matrix": prob_matrix,
    }


In [72]:
result = predict_match('Germany', 'Curaçao')


  Germany vs Curaçao
  Expected goals     : Germany 3.78 | Curaçao 0.60
  Predicted total    : 4.38 goals
  Most likely score  : 3-0
  Germany win      : 91.2%
  Draw               : 6.2%
  Curaçao win      : 2.6%


In [73]:
GROUPS = {
    "A": ["Mexico", "South Africa", "South Korea", "Czechia"],
    "B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Türkiye"],
    "E": ["Germany", "Curaçao", "Côte d'Ivoire", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "IR Iran", "New Zealand"],
    "H": ["Spain", "Cabo Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "Congo DR", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

In [82]:
from collections import defaultdict
import itertools
match_lambdas = {}

for group, teams in GROUPS.items():
    for team_a, team_b in itertools.combinations(teams, 2):
        X_row = build_feature_row(team_a, team_b, neutral=True, match_importance=3)
        lam_a = home_goal_model.predict(X_row)[0]
        lam_b = away_goal_model.predict(X_row)[0]
        match_lambdas[(team_a, team_b)] = (lam_a, lam_b)

print(f"Pre-computed {len(match_lambdas)} matchups")
print(match_lambdas)

Pre-computed 72 matchups
{('Mexico', 'South Africa'): (1.789858, 0.61777204), ('Mexico', 'South Korea'): (1.3342125, 1.1538969), ('Mexico', 'Czechia'): (1.7792745, 0.76562184), ('South Africa', 'South Korea'): (0.7809851, 1.7722511), ('South Africa', 'Czechia'): (1.0676929, 1.4215398), ('South Korea', 'Czechia'): (1.7691312, 0.9671216), ('Canada', 'Bosnia and Herzegovina'): (1.8368288, 0.5983526), ('Canada', 'Qatar'): (1.6906111, 0.55987453), ('Canada', 'Switzerland'): (1.1738249, 1.2602957), ('Bosnia and Herzegovina', 'Qatar'): (1.094945, 1.1257602), ('Bosnia and Herzegovina', 'Switzerland'): (0.6586707, 1.6424879), ('Qatar', 'Switzerland'): (0.7838063, 1.8131158), ('Brazil', 'Morocco'): (1.5956712, 1.4487315), ('Brazil', 'Haiti'): (2.1702013, 0.6977616), ('Brazil', 'Scotland'): (2.0795965, 0.71019536), ('Morocco', 'Haiti'): (1.9241453, 0.6670087), ('Morocco', 'Scotland'): (1.9655178, 0.677144), ('Haiti', 'Scotland'): (1.1159347, 1.3088982), ('United States', 'Paraguay'): (1.2025923, 

In [83]:
def simulate_single_match(team_a, team_b, rng, neutral=True, match_importance=3):
    lam_a, lam_b = match_lambdas[(team_a, team_b)]    

    ga = rng.poisson(lam_a)
    gb = rng.poisson(lam_b)

    return ga, gb

In [84]:


def simulate_group_once(teams, rng):
    pts = defaultdict(int)
    gf = defaultdict(int)
    ga = defaultdict(int)

    for team_a, team_b in itertools.combinations(teams, 2):
        g_a, g_b = simulate_single_match(team_a, team_b, rng)
        gf[team_a] += g_a
        gf[team_b] += g_b
        ga[team_a] += g_b
        gf[team_b] += g_a

        if g_a > g_b:
            pts[team_a] += 3
        elif g_a < g_b:
            pts[team_b] += 3
        else:
            pts[team_a] += 1
            pts[team_b] += 1

    gd = {t: gf[t] - ga[t] for t in teams}

    ranking = sorted(teams, key=lambda t:  (pts[t], gd[t], gf[t]), reverse=True)
    return ranking, pts, gd, gf


In [85]:
def run_monte_carlo(n_sims=5000, seed=42):
    rng = np.random.default_rng(seed)

    winner_count = defaultdict(int)
    runnerup_count = defaultdict(int)
    third_count = defaultdict(int)
    qualify_count = defaultdict(int)

    for sim in range(n_sims):
        third_candidates = []
        for group, teams in GROUPS.items():
            ranking, pts, gd, gf = simulate_group_once(teams, rng)
            winner_count[ranking[0]] += 1
            runnerup_count[ranking[1]] += 1
            qualify_count[ranking[0]] += 1
            qualify_count[ranking[1]] += 1

            third = ranking[2]
            third_candidates.append((third, pts[third], gd[third], gf[third]))

        # best 8 third placed teams advance across 12 groups
        third_candidates.sort(key=lambda x: (x[1], x[2], x[3]), reverse=True)
        for team, *_ in third_candidates[:8]:
            third_count[team] += 1
            qualify_count[team] += 1

    
    n = n_sims
    rows = []

    for group, teams in GROUPS.items():
        for t in teams:
            rows.append({
                'Group': group,
                'Team': t,
                'P(1st)':     round(winner_count[t]   / n * 100, 1),
                'P(2nd)':     round(runnerup_count[t] / n * 100, 1),
                'P(Best 3rd)':round(third_count[t]    / n * 100, 1),
                'P(Qualify)': round(qualify_count[t]  / n * 100, 1),
            })
    
    return pd.DataFrame(rows)

In [89]:
qualification_df = run_monte_carlo(n_sims=5000)

In [90]:
print('\n=== GROUP STAGE QUALIFICATION PROBABILITIES ===\n')
for group in sorted(GROUPS.keys()):
    group_df = qualification_df[qualification_df['Group'] == group].sort_values('P(Qualify)', ascending=False)
    print(f'--- Group {group} ---')
    print(group_df[['Team','P(1st)','P(2nd)','P(Best 3rd)','P(Qualify)']].to_string(index=False))
    print()


=== GROUP STAGE QUALIFICATION PROBABILITIES ===

--- Group A ---
        Team  P(1st)  P(2nd)  P(Best 3rd)  P(Qualify)
      Mexico    42.3    31.7         13.5        87.4
 South Korea    39.0    32.0         14.0        85.0
     Czechia    12.8    23.5         23.8        60.1
South Africa     5.9    12.8         14.4        33.2

--- Group B ---
                  Team  P(1st)  P(2nd)  P(Best 3rd)  P(Qualify)
           Switzerland    47.9    32.3         10.3        90.5
                Canada    39.7    35.6         12.2        87.5
                 Qatar     6.3    17.5         18.3        42.1
Bosnia and Herzegovina     6.1    14.6         16.7        37.5

--- Group C ---
    Team  P(1st)  P(2nd)  P(Best 3rd)  P(Qualify)
 Morocco    43.5    34.7         11.7        89.9
  Brazil    44.1    32.8         12.6        89.5
Scotland     7.6    18.7         24.8        51.1
   Haiti     4.8    13.7         20.9        39.4

--- Group D ---
         Team  P(1st)  P(2nd)  P(Best 3rd) 

In [91]:
print('=== OVERALL QUALIFICATION RANKING (all 48 teams) ===\n')
summary = qualification_df.sort_values('P(Qualify)', ascending=False).reset_index(drop=True)
print(summary[['Group','Team','P(1st)','P(2nd)','P(Best 3rd)','P(Qualify)']].to_string(index=False))

=== OVERALL QUALIFICATION RANKING (all 48 teams) ===

Group                   Team  P(1st)  P(2nd)  P(Best 3rd)  P(Qualify)
    H                  Spain    65.6    26.1          5.5        97.2
    J              Argentina    67.4    22.2          7.5        97.2
    L                England    52.3    31.6          9.9        93.9
    E                Ecuador    41.8    33.2         18.3        93.4
    E                Germany    38.9    34.3         18.7        92.0
    B            Switzerland    47.9    32.3         10.3        90.5
    K               Colombia    47.2    32.1         10.8        90.1
    C                Morocco    43.5    34.7         11.7        89.9
    C                 Brazil    44.1    32.8         12.6        89.5
    H                Uruguay    27.3    47.2         14.9        89.4
    G                IR Iran    44.2    29.2         15.0        88.5
    F                  Japan    45.1    31.0         12.3        88.4
    I                 France    47.2

In [92]:
import pickle

models = {
    'home_model': home_goal_model,
    'away_model': away_goal_model,
    'current_state': current_state
}

with open('wc2026_models.pkl', 'wb') as f:
    pickle.dump(models, f)
